# Previsão de safra de sequeiro no Ceará

Dá para prever, antes da colheita, quanto cada município do Ceará vai colher de milho e feijão? Quase toda essa
lavoura é de sequeiro e depende só da chuva da quadra chuvosa (fevereiro a maio). Quando a safra quebra, o
Garantia-Safra paga o agricultor familiar que perdeu 50% ou mais da produção, então saber cedo onde vai faltar tem uso
para o estado, para cooperativas e para quem planeja o abastecimento.

A base junta a produção municipal do IBGE (1990–2025) com a chuva diária dos postos da FUNCEME. O alvo é o
**rendimento em kg por hectare plantado**: usar a área plantada, e não a colhida, faz a perda aparecer.

Código que monta a base: [github.com/HowLuks/previsao-safra-ceara](https://github.com/HowLuks/previsao-safra-ceara).

In [ ]:
import glob
import json

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import PolyCollection
from sklearn.metrics import precision_score, recall_score


def achar(nome):
    """Caminho do arquivo no Kaggle (/kaggle/input) ou no repositório (dados/)."""
    for raiz in ("/kaggle/input", "dados", "../dados"):
        achados = glob.glob(f"{raiz}/**/{nome}", recursive=True)
        if achados:
            return achados[0]
    raise FileNotFoundError(nome)


df = pd.read_csv(achar("base.csv"), dtype={"cod_ibge": str}).sort_values(["cod_ibge", "produto", "ano"])
print(df.shape, df.ano.min(), df.ano.max(), df.cod_ibge.nunique(), "municípios")
df.head()

## Análise exploratória

O rendimento mediano dos municípios acompanha a chuva da quadra: as secas de 1993, 1998 e 2012–2016 aparecem como
vales. Mas a relação não é linear: 2009, o ano mais chuvoso, também teve rendimento baixo (chuva demais encharca e
atrapalha a colheita).

In [ ]:
print(df.groupby("produto").rendimento.describe().round(1))
print("\nperdas totais (produção zero):", (df.producao == 0).sum(), "linhas")

por_ano = df.groupby(["ano", "produto"]).rendimento.median().unstack()
chuva_ano = df.groupby("ano").chuva_quadra.median()
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(chuva_ano.index, chuva_ano, color="#9ecae1", label="chuva da quadra (mm, mediana)")
ax.set_ylabel("chuva fev–mai (mm)")
ax2 = ax.twinx()
por_ano.plot(ax=ax2, color={"milho": "#d95f02", "feijao": "#1b9e77"}, marker="o", ms=3, legend=False)
ax2.set_ylabel("rendimento (kg/ha plantado, mediana)")
ax.set_title("Chuva da quadra e rendimento mediano dos municípios")
fig.legend(loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08), frameon=False)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, produto in zip(axs, ["milho", "feijao"]):
    d = df[df.produto == produto]
    ax.scatter(d.chuva_quadra, d.rendimento, s=3, alpha=0.15, color="grey")
    faixas = d.groupby(pd.cut(d.chuva_quadra, range(0, 2001, 100)), observed=True).rendimento.median()
    ax.plot([f.mid for f in faixas.index], faixas.values, color="#d95f02", lw=2, label="mediana por faixa de 100 mm")
    ax.set(title=produto, xlabel="chuva fev–mai (mm)", xlim=(0, 2000))
axs[0].set_ylabel("rendimento (kg/ha plantado)")
axs[0].legend()
plt.show()

## Variáveis

Além da chuva, o modelo recebe o histórico do próprio município, calculado **só com anos anteriores** (nada do
futuro vaza para o passado): média de todos os anos, médias dos últimos 5 e 10 anos e rendimento do ano anterior. A
chuva também entra relativa ao normal do município (chuva do ano dividida pela média dos anos anteriores).

O alvo do modelo não é o rendimento direto, e sim o rendimento **dividido pela média histórica do município**: cada
município tem um nível muito diferente (o milho varia de quase zero a mais de 5 t/ha), e o que a chuva explica é
quanto o ano fica acima ou abaixo do normal dele. A previsão em kg/ha é essa razão vezes a média histórica.

Dois cenários de previsão:
- **fim de maio:** chuva de janeiro a maio (quadra completa);
- **fim de março:** só chuva de janeiro a março, um a dois meses antes. Aqui o modelo também pode receber os
  **índices do oceano** que a FUNCEME usa na previsão sazonal, já conhecidos em março: El Niño (Pacífico, dezembro a
  fevereiro) e o Atlântico tropical norte e sul (janeiro e fevereiro), além da **posição do município** (centro
  aproximado), porque o Cariri, no sul, chove diferente do sertão.
- **fim de março com previsão sazonal:** em vez dos índices do oceano, a **previsão de chuva para abril e maio do
  NMME** feita no começo de março (média de 7 modelos climáticos norte-americanos, grade de 1°), relativa ao normal
  de cada célula. O servidor público do CPC só tem as rodadas retroativas de 1991 a 2020, então esse cenário é
  avaliado em 2006–2020.

A área plantada do ano fica de fora: o IBGE só a publica depois da safra.

In [ ]:
g = df.groupby(["cod_ibge", "produto"]).rendimento
df["media_hist"] = g.transform(lambda s: s.shift().expanding().mean())
df["media_5anos"] = g.transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
df["media_10anos"] = g.transform(lambda s: s.shift().rolling(10, min_periods=1).mean())
df["rend_ano_anterior"] = g.shift()
for c in ("chuva_quadra", "chuva_fev_mar"):
    df[f"{c}_rel"] = df[c] / df.groupby(["cod_ibge", "produto"])[c].transform(lambda s: s.shift().expanding().mean())
df["milho"] = (df.produto == "milho").astype(int)

malha = json.load(open(achar("ce_municipios.geojson")))


def poligonos(geom):
    partes = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    return [np.array(p[0]) for p in partes]


# centro aproximado de cada município: média dos vértices do contorno
centro = {f["properties"]["codarea"]: np.concatenate(poligonos(f["geometry"])).mean(axis=0) for f in malha["features"]}
df["lon"] = df.cod_ibge.map(lambda c: centro[c][0])
df["lat"] = df.cod_ibge.map(lambda c: centro[c][1])

# NMME: chuva prevista para abril + maio, dividida pela média 1991–2020 da mesma célula; cada município usa a célula mais
# próxima do seu centro
nmme = pd.read_csv(achar("nmme_ceara.csv"))
nmme["abr_mai"] = nmme.nmme_abr + nmme.nmme_mai
nmme["relativo"] = nmme.abr_mai / nmme.groupby(["lat", "lon"]).abr_mai.transform("mean")
celulas = nmme[["lat", "lon"]].drop_duplicates().to_numpy()
celula = {c: tuple(celulas[((celulas - [la, lo]) ** 2).sum(axis=1).argmin()]) for c, (lo, la) in centro.items()}
relativo = nmme.set_index(["ano", "lat", "lon"]).relativo
df["nmme_abr_mai"] = [relativo.get((a, *celula[c]), np.nan) for c, a in zip(df.cod_ibge, df.ano)]
df["nmme_estado"] = df.ano.map(nmme.groupby("ano").relativo.mean())

HISTORICO = ["media_hist", "media_5anos", "media_10anos", "rend_ano_anterior", "milho"]
CHUVA_MARCO = ["chuva_jan", "chuva_fev", "chuva_mar", "chuva_fev_mar", "dias_chuva_fev_mar", "veranico_fev_mar",
               "chuva_fev_mar_rel"]
CHUVA_MAIO = CHUVA_MARCO + ["chuva_abr", "chuva_mai", "chuva_quadra", "dias_chuva_quadra", "veranico_quadra",
                            "chuva_quadra_rel"]
OCEANO = ["nino34_djf", "tna_jf", "tsa_jf", "dipolo_jf"]
LUGAR = ["lat", "lon"]
NMME = ["nmme_abr_mai", "nmme_estado"]

# o sinal do oceano no ano: correlação com o rendimento mediano do ano relativo ao normal de cada município
anomalia = (df.rendimento / df.media_hist).groupby(df.ano).median().loc[1991:]
print("correlação com o rendimento relativo do ano (1991–2025):")
print(df.groupby("ano")[OCEANO + ["nmme_estado"]].first().loc[1991:].corrwith(anomalia).round(2).to_string())

## Validação temporal

Para cada ano de 2006 a 2025, o modelo é treinado só com os anos anteriores e testado naquele ano, como seria usado
de verdade. Misturar os anos ao acaso daria um resultado bom demais e falso: o modelo veria a seca do mesmo ano em
outros municípios.

O **modelo de referência** prevê a média histórica do município. Se o LightGBM não bater isso, a chuva não está
ajudando.

Uma ressalva: a forma do alvo (razão sobre a média histórica, erro absoluto) foi escolhida olhando estes mesmos anos
de teste, entre poucas variantes. Prever o rendimento direto ficou pior que a referência.

In [ ]:
ANOS_TESTE = range(2006, 2026)


PARAMS = dict(objective="l1", n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=40,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=0, verbose=-1)


def treinar(treino, colunas, **extra):
    treino = treino[treino.media_hist > 0]
    # razão limitada a 4x a média: anos excepcionais isolados não puxam o modelo
    return lgb.LGBMRegressor(**PARAMS, **extra).fit(treino[colunas], (treino.rendimento / treino.media_hist).clip(upper=4))


def validar(colunas, anos=ANOS_TESTE):
    previsoes = []
    for ano in anos:
        teste = df[df.ano == ano]
        modelo = treinar(df[df.ano < ano], colunas)
        previsoes.append(pd.Series(modelo.predict(teste[colunas]) * teste.media_hist.values, index=teste.index))
    return pd.concat(previsoes).clip(lower=0)


teste = df[df.ano.isin(ANOS_TESTE)].copy()
teste["referencia"] = teste.media_hist
teste["maio"] = validar(HISTORICO + CHUVA_MAIO)
teste["marco"] = validar(HISTORICO + CHUVA_MARCO)
teste["marco_oceano"] = validar(HISTORICO + CHUVA_MARCO + OCEANO + LUGAR)
ANOS_NMME = range(2006, 2021)
teste["marco_nmme"] = validar(HISTORICO + CHUVA_MARCO + NMME + LUGAR, ANOS_NMME)  # vazio de 2021 em diante
print("linhas de teste sem histórico (fora da comparação):", teste.referencia.isna().sum())
teste = teste.dropna(subset=["referencia"])

MODELOS = {"referencia": "média histórica", "marco": "LightGBM até março", "marco_oceano": "LightGBM até março + oceano",
           "maio": "LightGBM até maio"}
erro = pd.DataFrame({nome: (teste[c] - teste.rendimento).abs().groupby(teste.produto).mean()
                     for c, nome in MODELOS.items()}).T
erro["geral"] = [(teste[c] - teste.rendimento).abs().mean() for c in MODELOS]
print("Erro médio absoluto (kg/ha), 2006–2025:")
print(erro.round(0))
for c in ("marco", "marco_oceano", "maio"):
    dif = erro.loc[MODELOS[c], "geral"] / erro.loc["média histórica", "geral"] - 1
    print(f"{MODELOS[c]}: erro {abs(dif):.0%} {'menor' if dif < 0 else 'maior'} que o da referência")

In [ ]:
por_ano = pd.DataFrame({nome: (teste[c] - teste.rendimento).abs().groupby(teste.ano).mean()
                        for c, nome in MODELOS.items()})
ax = por_ano.plot(figsize=(11, 4), marker="o", ms=3, color=["grey", "#b3b0d6", "#7570b3", "#d95f02"])
ax.set(title="Erro médio absoluto por ano de teste", ylabel="kg/ha", xlabel="")
plt.show()

A referência erra mais justamente nos anos de seca (2012–2016) e nos muito chuvosos: ela não sabe nada do ano. É
aí que a chuva faz diferença.

## Perda de 50% ou mais (critério do Garantia-Safra)

Conta como perda o município cujo rendimento ficou abaixo da metade da sua média histórica. A pergunta é: o modelo
aponta esses municípios antes?
- **precisão:** dos municípios apontados, quantos perderam de fato;
- **cobertura:** dos que perderam, quantos o modelo apontou.

In [ ]:
perda_real = teste.rendimento < 0.5 * teste.media_hist
print(f"municípios-ano com perda ≥ 50%: {perda_real.sum()} de {len(teste)} ({perda_real.mean():.0%})")
for c in ("marco", "marco_oceano", "maio"):
    aposta = teste[c] < 0.5 * teste.media_hist
    print(f"{MODELOS[c]}: precisão {precision_score(perda_real, aposta):.0%}, "
          f"cobertura {recall_score(perda_real, aposta):.0%}")

## Previsão sazonal do NMME (2006–2020)

Mesmo esquema, só nos anos em que há previsão do NMME. É a comparação mais importante para a previsão antecipada.

In [ ]:
t15 = teste[teste.ano.isin(ANOS_NMME)]
perda15 = t15.rendimento < 0.5 * t15.media_hist
ref15 = (t15.referencia - t15.rendimento).abs().mean()
MODELOS_15 = {**MODELOS, "marco_nmme": "LightGBM até março + NMME"}
for c, nome in MODELOS_15.items():
    e = (t15[c] - t15.rendimento).abs().mean()
    aposta = t15[c] < 0.5 * t15.media_hist
    texto_perda = "" if c == "referencia" else (f"   perda: precisão {precision_score(perda15, aposta):.0%}, "
                                                f"cobertura {recall_score(perda15, aposta):.0%}")
    print(f"{nome:30s} erro {e:5.0f} kg/ha ({e / ref15 - 1:+.0%}){texto_perda}")

por_ano15 = pd.DataFrame({nome: (t15[c] - t15.rendimento).abs().groupby(t15.ano).mean()
                          for c, nome in MODELOS_15.items() if c in ("referencia", "maio", "marco_nmme")})
ax = por_ano15.plot(figsize=(11, 4), marker="o", ms=3, color=["grey", "#d95f02", "#1b9e77"])
ax.set(title="Erro médio absoluto por ano, 2006–2020", ylabel="kg/ha", xlabel="")
plt.show()

## O que o modelo usa

Importância das variáveis (ganho) no modelo de maio treinado com todos os anos.

In [ ]:
final = treinar(df, HISTORICO + CHUVA_MAIO, importance_type="gain")
imp = pd.Series(final.feature_importances_, HISTORICO + CHUVA_MAIO).sort_values()
ax = (imp / imp.sum()).plot.barh(figsize=(7, 5), color="#d95f02")
ax.set(title="Importância das variáveis (fração do ganho)", xlabel="")
plt.show()

## Mapa: 2012, primeiro ano da grande seca

Rendimento de milho real e previsto (até maio e até março com o NMME) por município, num ano que o modelo não viu no
treino.

In [ ]:
ANO_MAPA = 2012
d = teste[(teste.ano == ANO_MAPA) & (teste.produto == "milho")].set_index("cod_ibge")
fig, axs = plt.subplots(1, 3, figsize=(15, 6))
for ax, (coluna, titulo) in zip(axs, [("rendimento", "real"), ("maio", "previsto até maio"),
                                      ("marco_nmme", "previsto até março (com NMME)")]):
    polis, valores = [], []
    for f in malha["features"]:
        for p in poligonos(f["geometry"]):
            polis.append(p)
            valores.append(d[coluna].get(f["properties"]["codarea"], np.nan))
    col = PolyCollection(polis, array=np.ma.masked_invalid(valores), cmap="YlGn", edgecolor="white", lw=0.3)
    col.set_clim(0, 800)
    col.cmap.set_bad("#dddddd")
    ax.add_collection(col)
    ax.autoscale()
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"Milho {ANO_MAPA}: {titulo}")
fig.colorbar(col, ax=axs, shrink=0.7, label="kg/ha plantado (cinza: sem dado)")
plt.show()

## Conclusões

- **Com a chuva até maio, o modelo erra 12% menos que a média histórica** (212 contra 239 kg/ha, 2006–2025). O ganho
  vem dos anos fora do normal, justamente os de seca.
- **Até março, só a chuva não basta:** o modelo empata com a média histórica. Os índices do oceano (El Niño,
  Atlântico) ajudam pouco (erro 3% menor): o sinal tem o sentido esperado, mas é fraco (correlação em torno de −0,25
  com o rendimento do ano).
- **A previsão sazonal do NMME resolve boa parte disso.** Em 2006–2020, o modelo de março com o NMME erra 23% menos
  que a média histórica, um pouco melhor que o de maio no mesmo período (20%), e encontra cerca de um terço das
  perdas de 50% ou mais, acertando dois terços das vezes que aponta. Nas secas de 2012 e 2016 o erro cai a menos da
  metade. O NMME tem correlação de cerca de 0,5 com o rendimento do ano, o dobro dos índices do oceano: os modelos
  climáticos já juntam o Pacífico e o Atlântico do jeito certo.

Limitações:
- o NMME vem de rodadas retroativas (os modelos rodados depois, com os dados disponíveis em cada março); a previsão
  real de 2021 em diante não está no servidor público, então não foi testada;
- o normal de cada célula do NMME usa 1991–2020 inteiro;
- a forma do alvo e as variáveis foram escolhidas olhando os anos de teste, entre poucas variantes;
- o rendimento do IBGE é uma estimativa municipal, e a chuva é a média dos postos de cada município.